In [ ]:
#A. EXTRACT

# 1. Baca transaksi (CSV)
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
print(f"Jumlah baris df_transaksi: {df_transaksi.count()}")
df_transaksi.printSchema()

# 2. Baca produk (JSON Lines)
df_produk = spark.read.json("tugas6_produk.json")
print(f"Jumlah baris df_produk: {df_produk.count()}")
df_produk.printSchema()

# 3. Baca ulasan (CSV)
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
print(f"Jumlah baris df_ulasan: {df_ulasan.count()}")
df_ulasan.printSchema()

In [ ]:
# B & C. TRANSFORM — Penggabungan, Penanganan Data Kosong & Pengayaan

# 1. Join transaksi dengan ulasan menggunakan LEFT JOIN (karena tidak semua transaksi memiliki ulasan)
df_joined_1 = df_transaksi.join(df_ulasan, on="order_id", how="left")

# 2. Join dengan produk menggunakan INNER JOIN (karena setiap transaksi pasti memiliki produk yang valid)
df_joined_2 = df_joined_1.join(df_produk, on="product_id", how="inner")

# 3. Tambahkan kolom 'ada_ulasan' bernilai True/False SEBELUM na.fill()
#    'ada_ulasan' True jika kolom 'rating' TIDAK NULL
df_transformed = df_joined_2.withColumn("ada_ulasan", F.col("rating").isNotNull())

# 4. Hitung total_pendapatan = unit_terjual * harga
df_transformed = df_transformed.withColumn("total_pendapatan", F.col("unit_terjual") * F.col("harga"))

# 5. Isi nilai rating yang kosong (null) dengan angka 0
df_final = df_transformed.fillna({"rating": 0})

# Tampilkan beberapa baris hasil transformasi
df_final.show(10)

In [ ]:
# D. LOAD

# Ganti [username] dengan username HDFS / sistem Anda
hdfs_path = "/user/[username]/tugas6/hasil_etl"

# Simpan ke HDFS format Parquet dipartisi berdasarkan kategori
df_final.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)

# Verifikasi dengan terminal command
!hdfs dfs -ls -R /user/[username]/tugas6/hasil_etl

# Baca kembali data yang tersimpan di HDFS dan tampilkan count()
df_hdfs_check = spark.read.parquet(hdfs_path)
print(f"Jumlah baris tersimpan di HDFS: {df_hdfs_check.count()}")

In [ ]:
# E. Insight Akhir

# Menghitung total transaksi dan jumlah ulasan per kategori
df_insight = df_final.groupBy("kategori").agg(
    F.count("order_id").alias("total_transaksi"),
    F.sum(F.when(F.col("ada_ulasan") == True, 1).otherwise(0)).alias("jumlah_dengan_ulasan")
)

# Menambahkan persentase ulasan
df_insight = df_insight.withColumn(
    "persentase_ulasan", 
    (F.col("jumlah_dengan_ulasan") / F.col("total_transaksi")) * 100
).orderBy("persentase_ulasan")

df_insight.show()